#### Решение двумерного нестационарного уравнения Навье-Стокса для ламинарного течения несжимаемой жидкости в канале (течение Пуазёйля)

По мотивам статьи (Rao) Physics-informed deep learning for incompressible laminar flows

**Уравнение Навье-Стокса для несжимаемой жидкости, дополненое уравнением неразрывности:**

$
\begin{cases}
    \dfrac{\partial \vec{v}}{\partial t} + \left( \vec{v} \cdot \nabla \right) \vec{v} = \dfrac{1}{\rho} \nabla \cdot \sigma + \vec{f},  \\
    \nabla \cdot \vec{v} = 0,
\end{cases}
$

где
- $\vec{v} = (u, \mathrm{v})^T$ — векторное поле скоростей,
- $\rho = \mathrm{const}$ — плотность жидкости,
- $\sigma = -p I + \mu \left[ \nabla \vec{v} + (\nabla \vec{v})^T \right]$ — тензор напряжений Коши, $\left[ \sigma_{ij} \right] = M / (L \cdot T^2)$,
- $\mu$ — коэффициент динамической вязкости, $\left[ \mu \right]$ = $M / (L \cdot T)$,
- $\vec{f}$ — сила на единицу массы, $\left[ f \right]$ = $L / T^2$.

<br>

**Обезразмеривание:**

- $\widetilde{L}$ — характерная длина,
- $\widetilde{U}$ — характерная скорость,
- $\widetilde{T} = \widetilde{L} / \widetilde{U}$ — характерное время,
- $\widetilde{\Sigma} = \rho \widetilde{U}^2$
- $\widetilde{F} = \widetilde{U}^2 / \widetilde{L}$ — характерная сила на единицу массы,
- $\widetilde{P} = \rho \widetilde{U}^2$ — характерное давление,
- $\mathrm{Re} = \rho \widetilde{U} \widetilde{L} / \mu $ — число Рейнольдса


Обезразмеренное уравнение Навье-Стокса для несжимаемой жидкости, дополненое уравнением неразрывности:

$
\begin{cases}
    \dfrac{\partial \vec{\widetilde{v}}}{\partial \tilde{t}} + \left( \vec{\widetilde{v}} \cdot \nabla \right) \vec{\widetilde{v}} = \nabla \cdot \widetilde{\sigma} + \vec{\widetilde{f}},  \\
    \nabla \cdot \vec{\widetilde{v}} = 0,
\end{cases}
$

где $\widetilde{\sigma} = -\widetilde{p} I + \dfrac{1}{\mathrm{Re}} \left[ \nabla \vec{\widetilde{v}} + (\nabla \vec{\widetilde{v}})^T \right]$

<br>

**Постановка задачи:**

![alt text](pictures/navier-stokes_2d_incompressible_nonsteady_obstacle.png)

$
\begin{cases}
    \dfrac{\partial \vec{v}}{\partial t} + \left( \vec{v} \cdot \nabla \right) \vec{v} - \dfrac{1}{\rho} \nabla \cdot \sigma = 0,  \\
    \nabla \cdot \vec{v} = 0,
\end{cases}
$

Начальное и краевые условия:
- $\left. \vec{v} \right|_{t=t_{\mathrm{min}}} = \left. p \right|_{t=t_{\mathrm{min}}} = 0$ — начальное условие,
- $\left. \vec{v} \right|_{\Gamma_1} = \left. \vec{v} \right|_{\Gamma_2} = \left. \vec{v} \right|_{\Gamma_5} = 0$ — условия прилипания,
- $\left. u \right|_{\Gamma_3} = - \dfrac{U_\mathrm{max}}{h^2} (y - y_\mathrm{min}) (y - y_\mathrm{max}) \cdot \mathrm{step(t - 0{,}05 \; с)}$ — профиль течения Пуазейля,
- $\left. \sigma \cdot \vec{n} \right|_{\Gamma_4} = 0$.

Здесь:
- $U_\mathrm{max}$ — задаваемый параметр;
- $h = (y_\mathrm{max} - y_\mathrm{min}) / 2$;
- $
  \mathrm{step}(t) =
  \begin{cases}
  0, & t < d, \\
  0.5 + \dfrac{15}{16}\dfrac{t}{d} - \dfrac{5}{8}\left(\dfrac{t}{d}\right)^3 + \dfrac{3}{16}\left(\dfrac{t}{d}\right)^5, & -d \le t \le d, \\
  1, & t > d;
  \end{cases}
  $
  
  где $d = 0{,}05$ — половина ширины переходной зоны;
- $\Gamma_1$, $\Gamma_2$ — боковые стенки трубы;
- $\Gamma_3$, $\Gamma_4$ — вход в трубу и выход из неё соответственно;
- $\Gamma_5$ — препятствие диаметром $d_\mathrm{obs} = 0{,}5 h$ с координатами центра $(x_\mathrm{obs}, y_\mathrm{obs}) = (x_\mathrm{min}, y_\mathrm{min}) + (1{,}25 h, h)$.
- $\vec{n}$ — внешняя нормаль

Задача решается относительно скалярной функции тока $\psi(x,y)$, такой что $u = \partial_y \psi$, $v = -\partial_x \psi$

In [1]:
from pathlib import Path
import os

# Переместиться в папку с блокнотом, создать папки для графиков и прочего
notebook_path = Path("/content/drive/MyDrive/Colab Notebooks/NN_pytorch_BVP")
if os.getenv("COLAB_RELEASE_TAG"):    # Запущен ли блокнот в Google Colab
    is_in_Colab = True
    from google.colab import drive
    drive.mount('/content/drive')
    notebook_path.mkdir(parents=True, exist_ok=True)
    os.chdir(notebook_path)
else:
    is_in_Colab = False

In [ ]:
import time
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, replace, asdict, field, fields
import math
from contextlib import nullcontext
import json
import os
import shutil
import copy
from types import MappingProxyType
import warnings

import torch
import torch.optim as optim
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.animation as anim
import matplotlib.ticker as mticker
import matplotlib.colors as colors
import matplotlib.tri as tri
import matplotlib as mpl
import numpy as np
from tqdm import tqdm, trange
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator

from NN_pytorch_BVP.models import MultilayerPerceptronWithFFE, compute_grad_theta_norm, initialize_weights
from NN_pytorch_BVP.sampling import sample_points_3D
from NN_pytorch_BVP.utils import update_axis_limits_hysteresis, compute_grad, get_uv_from_psi
from NN_pytorch_BVP.utils import step_smoothed, get_trainable_layers, log_per_layer_ms, make_cosine_annealing_warmup_scheduler

mpl.rcParams.update({
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "grid.alpha": 1.0,   # optional
})

torch.manual_seed(2008)

def weighted_subsample(
    points: torch.Tensor,    # shape (n, 2)
    weights: torch.Tensor,   # shape (n, 1) or (n,)
    k: int,
    seed: int | None = None,
):
    w = weights.squeeze(-1)   # shape (n,)

    g = None
    if seed is not None:
        g = torch.Generator(device=points.device)
        g.manual_seed(seed)

    idx = torch.multinomial(w, num_samples=k, replacement=False, generator=g)
    return points[idx]

def cfg_field(group: str, **kwargs):
    """
    Create a dataclass field with group metadata.

    All keyword arguments except `group` are passed directly to
    dataclasses.field().
    """
    metadata = kwargs.pop("metadata", None)
    if metadata is None:
        metadata = {}
    else:
        metadata = dict(metadata)
    metadata["group"] = group

    return field(metadata=metadata, **kwargs)

@dataclass(frozen=True, slots=True)
class PINNConfig:
    # General
    results_folder: Path = Path.cwd() / 'runs' / "05_03_navier-stokes_2d_incompressible_nonsteady_obstacle"
    reference_solution_path: Path = Path.cwd() / "data" / "navier-stokes_2d_incompressible_nonsteady_obstacle.pt"

    # Initial-condition source:
    # - "zero": use zero velocity and pressure as IC
    # - "reference": use COMSOL/reference solution at ic_reference_time
    # - "pretrained_model": use a saved model as IC target
    ic_source: str = "zero"
    ic_reference_time: float | None = 4.0    # Reference-solution time, in seconds, used as the IC target when ic_source="reference"
    pretrained_ic_model_path: Path | None = None    # Path to a pretrained IC model. Used when ic_source="pretrained_model"

    # Problem parameters
    xlims: tuple[float, float] = cfg_field(default=(-3.0, -1.0), group="Problem parameters")    # x_min, x_max (метры)
    ylims: tuple[float, float] = cfg_field(default=(-0.1, 0.4), group="Problem parameters")    # y_min, y_max (метры)
    tlims: tuple[float, float] = cfg_field(default=(0.0, 0.35), group="Problem parameters")    # t_min, t_max (секунды)
    coords_obs: tuple[float, float] = cfg_field(init=False, group="Problem parameters")    # x_obs, y_obs
    d_obs: float = cfg_field(init=False, group="Problem parameters")
    nu: float = cfg_field(default=0.00103, group="Problem parameters")    # кинематическая вязкость (масло касторовое), м2/с
    rho: float = cfg_field(default=960, group="Problem parameters")    # плотность (масло касторовое), кг/м3
    u0: float = cfg_field(default=1.0, group="Problem parameters")    # начальная скорость в центре потока, м/с

    # Dimensionless parameters
    L_wave: float = cfg_field(init=False, group="Dimensionless parameters")    # характерная длина, м
    U_wave: float = cfg_field(init=False, group="Dimensionless parameters")    # характерная скорость, м/с
    T_wave: float = cfg_field(init=False, group="Dimensionless parameters")    # характерное время, с
    P_wave: float = cfg_field(init=False, group="Dimensionless parameters")    # характерное давление, Па
    F_wave: float = cfg_field(init=False, group="Dimensionless parameters")    # характерная сила на единицу массы, Н/кг
    Re: float = cfg_field(init=False, group="Dimensionless parameters")    # число Рейнольдса
    Sigma_wave: float = cfg_field(init=False, group="Dimensionless parameters")    # характерное механическое напряжение, Н/м2

    # Model parameters
    device: str = cfg_field(default="cuda" if torch.cuda.is_available() else "cpu", group="Model parameters")
    init_scheme: str = cfg_field(default="glorot_normal", group="Model parameters")
    layers: tuple[int, ...] = cfg_field(default=(3, 256, 256, 5), group="Model parameters")   # (t, x, y) -> (psi, p, sigma_xx, sigma_xy(=sigma_yx), sigma_yy)
    activation: str = cfg_field(default="tanh", group="Model parameters")

    # Training parameters
    n_epochs: int = cfg_field(default=5000, group="Training parameters")
    n_batches: int = cfg_field(default=10, group="Training parameters")    # number of batches per epoch
    use_data_loss: bool = cfg_field(default=False, group="Training parameters")
    lambdas: MappingProxyType = cfg_field(default_factory=lambda: MappingProxyType(dict(    # MappingProxyType is a read-only dictionary view
        u_r=1.0, v_r=1.0, sigma_xx_r=1.0, sigma_xy_r=1.0, sigma_yy_r=1.0, p_r=1.0,
        u_ic=1.0, v_ic=1.0, p_ic=1.0,
        u_walls=1.0, v_walls=1.0, u_in=1.0, v_in=1.0,
        sigma_xx_out=1.0, sigma_xy_out=1.0,
        u_data=1.0, v_data=1.0)), group="Training parameters"
    )
    lr: float = cfg_field(default=1e-3, group="Training parameters")    # initial learning rate
    logging_freq: int = cfg_field(default=1, group="Training parameters")    # log metrics once every N gradient descent steps

    # Sampling
    use_static_training_set: bool = cfg_field(default=False, group="Sampling parameters")
    use_adaptive_sampling: bool = cfg_field(default=False, group="Sampling parameters")
    resampling_frequency: int = cfg_field(default=1, group="Sampling parameters")    # resample collocation points once every N gradient descent steps

    n_r: int = cfg_field(default=80000, group="Sampling parameters")
    n_ic: int = cfg_field(init=False, group="Sampling parameters")
    n_bc_xmin: int = cfg_field(init=False, group="Sampling parameters")
    n_bc_xmax: int = cfg_field(init=False, group="Sampling parameters")
    n_bc_ymin: int = cfg_field(init=False, group="Sampling parameters")
    n_bc_ymax: int = cfg_field(init=False, group="Sampling parameters")
    n_bc_obs: int = cfg_field(init=False, group="Sampling parameters")
    n_data: int = cfg_field(init=False, group="Sampling parameters")
    data_fraction: float = cfg_field(default=0.1, group="Sampling parameters")

    batch_size_r: int = cfg_field(init=False, group="Sampling parameters")
    batch_size_ic: int = cfg_field(init=False, group="Sampling parameters")
    batch_size_bc_xmin: int = cfg_field(init=False, group="Sampling parameters")
    batch_size_bc_xmax: int = cfg_field(init=False, group="Sampling parameters")
    batch_size_bc_ymin: int = cfg_field(init=False, group="Sampling parameters")
    batch_size_bc_ymax: int = cfg_field(init=False, group="Sampling parameters")
    batch_size_bc_obs: int = cfg_field(init=False, group="Sampling parameters")
    batch_size_data: int = cfg_field(init=False, group="Sampling parameters")

    # Log metrics (like loss and error) during training for different t intervals
    use_temporal_logging: bool = False
    temporal_logging_freq: int = 200    # log metrics once every N gradient descent steps
    temporal_logging_nbins: int = 10    # Slice time into this number of equal segments

    # Checkpointing
    use_checkpointing: bool = cfg_field(default=False, group="Checkpointing")
    checkpointing_freq: int = cfg_field(default=2000, group="Checkpointing")    # save model and metrics to the results_folder once every N gradient descent steps

    # Fourier Feature embedding
    use_FFE: bool = cfg_field(default=False, group="Fourier Feature embedding")
    FFE_m: int = cfg_field(default=100, group="Fourier Feature embedding")
    FFE_sigma: float = cfg_field(default=1.0, group="Fourier Feature embedding")
    FFE_keep_dims: None | tuple[int, ...] = cfg_field(default=None, group="Fourier Feature embedding")    # Indices of input dimensions to be passed through unchanged

    # Grad norm weighting
    use_grad_norm_weighting: bool = cfg_field(default=False, group="Grad norm weighting")
    grad_norm_weighting_alpha: float = cfg_field(default=0.9, group="Grad norm weighting")
    grad_norm_weighting_freq: int = cfg_field(default=200, group="Grad norm weighting")   # update lambdas once every N gradient descent steps

    # Causal weighting
    use_causal_weighting_scheme: bool = cfg_field(default=False, group="Causal weighting")
    causal_m: int = cfg_field(default=10, group="Causal weighting")
    causal_eps: float | tuple[float, ...] = cfg_field(default=100.0, group="Causal weighting")
    W_logging_freq: int = cfg_field(default=100, group="Causal weighting")    # log temporal weights and L_t once every N gradient descent steps

    # Time marching
    use_time_marching: bool = cfg_field(default=False, group="Time marching")
    n_time_windows: int = cfg_field(default=10, group="Time marching")

    # Video parameters
    render_video: bool = cfg_field(default=False, group="Video parameters")
    video_fps: int = cfg_field(default=30, group="Video parameters")
    video_dpi: int = cfg_field(default=100, group="Video parameters")
    video_render_freq: int = cfg_field(default=100, group="Video parameters")    # render a frame once every N gradient descent steps

    def to_grouped_dict(self, ungrouped_name: str = "General") -> dict:
        grouped = {}
        ungrouped = {}

        for f in fields(self):
            value = getattr(self, f.name)
            if isinstance(value, MappingProxyType):
                value = dict(value)
            group = f.metadata.get("group")

            if group is None:
                ungrouped[f.name] = value
            else:
                grouped.setdefault(group, {})
                grouped[group][f.name] = value

        result = {}
        if ungrouped:
            result[ungrouped_name] = ungrouped
        result.update(grouped)

        return result

    def __str__(self):
        config_dict = self.to_grouped_dict()

        lines = []
        for group_name, group_dict in config_dict.items():
            lines.append(group_name.upper())
            for key, value in group_dict.items():
                lines.append(f"{key}: {value}")
            lines.append("")
        return "\n".join(lines[:-1])

    def __post_init__(self):
        allowed_ic_sources = {"zero", "reference", "pretrained_model"}
        if self.ic_source not in allowed_ic_sources:
            raise ValueError(f"Unknown ic_source={self.ic_source!r}. Expected one of {sorted(allowed_ic_sources)}.")
        if self.ic_source == "reference" and self.ic_reference_time is None:
            raise ValueError("ic_reference_time must be provided when ic_source='reference'.")
        if self.ic_source == "pretrained_model" and self.pretrained_ic_model_path is None:
            raise ValueError("pretrained_ic_model_path must be provided when ic_source='pretrained_model'.")

        xlims = self.xlims[:]
        ylims = self.ylims[:]
        tlims = self.tlims[:]

        h = (ylims[1] - ylims[0])/2
        coords_obs = (xlims[0] + 1.25*h, ylims[0] + h)
        d_obs = 0.5*h
        object.__setattr__(self, "coords_obs", coords_obs)
        object.__setattr__(self, "d_obs", d_obs)

        # n_r ~ Lx*Ly*T    =>       h_r ~ (Lx*Ly*T/n_r)**(1/3)
        # n_ic ~ Lx*Ly     =>      h_ic ~ (Lx*Ly/n_ic)**(1/2)
        # n_bc_xmin ~ Ly*T => h_bc_xmin ~ (Ly*T/n_bc_xmin)**(1/2)
        # n_bc_ymin ~ Lx*T => h_bc_ymin ~ (Lx*T/n_bc_ymin)**(1/2)
        # n_obs ~ pi*d*T   =>     h_obs ~ (pi*d*T/n_obs)**(1/2)
        #
        # n_r ~ n_ic      =>      n_ic ~  Lx*Ly*[n_r/(Lx*Ly*T)]**(2/3)
        # n_r ~ n_bc_xmin => n_bc_xmin ~   Lx*T*[n_r/(Lx*Ly*T)]**(2/3)
        # n_r ~ n_bc_ymin => n_bc_ymin ~   Ly*T*[n_r/(Lx*Ly*T)]**(2/3)
        # n_r ~ n_obs     =>     n_obs ~ pi*d*T*[n_r/(Lx*Ly*T)]**(2/3)
        Lx = xlims[1] - xlims[0]
        Ly = ylims[1] - ylims[0]
        T = tlims[1] - tlims[0]
        tmp = (self.n_r/(Lx*Ly*T))**(2.0/3.0)
        n_ic = math.ceil(Lx*Ly*tmp)
        n_bc_xmin = math.ceil(Ly*T*tmp)
        n_bc_xmax = n_bc_xmin
        n_bc_ymin = math.ceil( Lx*T*tmp )
        n_bc_ymax = n_bc_ymin
        n_bc_obs = 25 * math.ceil(math.pi*d_obs*T*tmp)

        object.__setattr__(self, "n_ic", n_ic)
        object.__setattr__(self, "n_bc_xmin", n_bc_xmin)
        object.__setattr__(self, "n_bc_xmax", n_bc_xmax)
        object.__setattr__(self, "n_bc_ymin", n_bc_ymin)
        object.__setattr__(self, "n_bc_ymax", n_bc_ymax)
        object.__setattr__(self, "n_bc_obs", n_bc_obs)
        n_data = math.ceil(self.n_r * self.data_fraction) if self.use_data_loss else 0
        object.__setattr__(self, "n_data", n_data)

        lambdas = dict(self.lambdas)
        if not self.use_data_loss:
            for key in ["u_data", "v_data", "p_data"]:
                lambdas.pop(key, None)
        object.__setattr__(self, "lambdas", MappingProxyType(lambdas))

        counts_dict = {
            "r": self.n_r,
            "ic": n_ic,
            "bc_xmin": n_bc_xmin,
            "bc_xmax": n_bc_xmax,
            "bc_ymin": n_bc_ymin,
            "bc_ymax": n_bc_ymax,
            "bc_obs": n_bc_obs,
        }
        if self.use_data_loss:
            counts_dict["data"] = self.n_data
        else:
            object.__setattr__(self, "batch_size_data", 0)
        for name, n in counts_dict.items():
            batch_size = n / self.n_batches
            if batch_size < 1:
                raise ValueError(
                    f"batch_size_{name} turned out to be less than 1 ({batch_size:.1f}). "
                    "Increase n_r or decrease n_batches!"
                )
            object.__setattr__(self, f"batch_size_{name}", math.ceil(batch_size))

        # Обезразмеривание
        object.__setattr__(self, "L_wave", d_obs)
        object.__setattr__(self, "U_wave", 2.0*self.u0/3.0)
        object.__setattr__(self, "T_wave", self.L_wave / self.U_wave)
        object.__setattr__(self, "P_wave", self.rho * self.U_wave**2)
        object.__setattr__(self, "F_wave", self.U_wave / self.T_wave)
        object.__setattr__(self, "Re", self.U_wave * self.L_wave / self.nu)
        object.__setattr__(self, "Sigma_wave", self.rho * self.U_wave**2)

class Trainer:
    def _load_pretrained_ic_model(self):
        cfg = self.cfg
        if cfg.pretrained_ic_model_path is None:
            return None
        else:
            pretrained_ic_model = MultilayerPerceptronWithFFE.load(
                cfg.pretrained_ic_model_path,
                device=cfg.device,
            ).eval()
            for p in pretrained_ic_model.parameters():
                p.requires_grad_(False)
        return pretrained_ic_model

    def __init__(self, model, optimizer_factory, scheduler_factory, cfg: PINNConfig):
        self.model = model.to(torch.device(cfg.device))
        self.optimizer_factory = optimizer_factory
        self.scheduler_factory = scheduler_factory
        self.optimizer = optimizer_factory(self.model)
        self.scheduler = scheduler_factory(self.optimizer)
        self.cfg = cfg

        self.results_folder = self.cfg.results_folder
        if self.results_folder.exists():
            shutil.rmtree(self.results_folder)
        self.results_folder.mkdir(parents=True, exist_ok=True)

        if cfg.use_checkpointing or cfg.use_time_marching:
            (self.results_folder / "model").mkdir(parents=True, exist_ok=True)
        if cfg.use_checkpointing:
            (self.results_folder / "metrics").mkdir(parents=True, exist_ok=True)

        # Save cfg dataclass object to the results folder as json
        config_dict = cfg.to_grouped_dict(ungrouped_name="General")
        #del config_dict["General"]["results_folder"]
        (self.results_folder / "config.json").write_text(json.dumps(config_dict, indent=4, ensure_ascii=False, default=str))

        self.xlims = [cfg.xlims[0]/cfg.L_wave, cfg.xlims[1]/cfg.L_wave]
        self.ylims = [cfg.ylims[0]/cfg.L_wave, cfg.ylims[1]/cfg.L_wave]
        self.tlims = [cfg.tlims[0]/cfg.T_wave, cfg.tlims[1]/cfg.T_wave]
        self.coords_obs = [cfg.coords_obs[0]/cfg.L_wave, cfg.coords_obs[1]/cfg.L_wave]
        self.d_obs = cfg.d_obs/self.cfg.L_wave
        self.u0 = cfg.u0/cfg.U_wave
        self.h = (cfg.ylims[1] - cfg.ylims[0]) / 2.0 / cfg.L_wave
        self.nu = cfg.nu
        self.rho = cfg.rho
        self.device = cfg.device
        self.use_static_training_set = cfg.use_static_training_set
        self.use_adaptive_sampling = cfg.use_adaptive_sampling
        self.use_data_loss = cfg.use_data_loss

        self.n_r, self.n_ic = cfg.n_r, cfg.n_ic
        self.n_bc_xmin, self.n_bc_xmax = cfg.n_bc_xmin, cfg.n_bc_xmax
        self.n_bc_ymin, self.n_bc_ymax = cfg.n_bc_ymin, cfg.n_bc_ymax
        self.n_bc_obs = cfg.n_bc_obs

        # Log metrics (like loss and error) during training for different t intervals
        self.use_temporal_logging = cfg.use_temporal_logging
        if self.use_temporal_logging:
            raise NotImplementedError("Temporal logging is currently not implemented.")
        self.temporal_logging_freq = cfg.temporal_logging_freq
        self.temporal_logging_nbins = cfg.temporal_logging_nbins
        self.temporal_logging_edges = torch.linspace(
            self.tlims[0], self.tlims[1], self.temporal_logging_nbins + 1, device=self.device
        )

        # Идеология работы с безразмерными величинами следующая: каждый метод класса Train
        # работает в безразмерных единицах. Все величины, которые подразумевают обезразмеривание,
        # обезразмериваются. В конфиге cfg всё остаётся размерным
        self.L_wave = cfg.L_wave
        self.U_wave = cfg.U_wave
        self.T_wave = cfg.T_wave
        self.P_wave = cfg.P_wave
        self.Re_inv = 1.0 / cfg.Re

        self.grad_norm_weighting_alpha = cfg.grad_norm_weighting_alpha
        self.use_causal_weighting_scheme = cfg.use_causal_weighting_scheme
        self.causal_m = cfg.causal_m

        if isinstance(cfg.causal_eps, (int, float)):
            self.causal_eps_schedule = [float(cfg.causal_eps)]
        else:
            self.causal_eps_schedule = [float(eps) for eps in cfg.causal_eps]
        self.causal_eps = self.causal_eps_schedule[0]

        self.use_time_marching = cfg.use_time_marching
        self.n_time_windows = cfg.n_time_windows

        # Import reference solution
        ref_solution_dict = torch.load(cfg.reference_solution_path, map_location=self.device)
        self.xy_ref = ref_solution_dict["xy"] / self.L_wave    # shape (n, 2), where each row is a grid point coordinate
        self.t_ref = ref_solution_dict["t"] / self.T_wave    # shape (m), distinct time moments
        self.u_ref = ref_solution_dict["u"] / self.U_wave    # shape (n, m), where i-th column is u field at t[i]
        self.v_ref = ref_solution_dict["v"] / self.U_wave
        self.p_ref = ref_solution_dict["p"] / self.P_wave
        self.cl_ref = ref_solution_dict["C_L"]
        self.cd_ref = ref_solution_dict["C_D"]

        # Initial conditions handling
        self.ic_source = cfg.ic_source
        self.ic_reference_time = None if cfg.ic_reference_time is None else cfg.ic_reference_time / self.T_wave
        self.pretrained_ic_model = self._load_pretrained_ic_model()
        if self.ic_source == "reference":
            u_ref, v_ref, p_ref = self._ref_uvp_at_time(self.ic_reference_time)
            xy_np = self.xy_ref.detach().cpu().numpy()
            self.ic_interp_u = LinearNDInterpolator(xy_np, u_ref.detach().cpu().numpy())
            self.ic_interp_v = LinearNDInterpolator(xy_np, v_ref.detach().cpu().numpy())
            self.ic_interp_p = LinearNDInterpolator(xy_np, p_ref.detach().cpu().numpy())
        else:
            self.ic_interp_u, self.ic_interp_v, self.ic_interp_p = None, None, None

        self.causal_edges = torch.linspace(
            self.tlims[0], self.tlims[1], self.causal_m + 1, device=self.device
        )

        self.logged_layers, self.logged_layer_names = get_trainable_layers(model)

        # Build one fixed COMSOL data sample.
        if self.use_data_loss:
            n_data = self.cfg.n_data

            txy_all, uvp_all = self.assemble_reference_txy_uvp(self.t_ref)

            if n_data > txy_all.shape[0]:
                raise ValueError(
                    f"Requested n_data={n_data}, but only {txy_all.shape[0]} "
                    "COMSOL data points are available after assembling reference data."
                )

            g = torch.Generator(device="cpu")
            #g.manual_seed(self.cfg.data_seed)

            idx = torch.randperm(txy_all.shape[0], generator=g)[:n_data].to(self.device)

            self.txy_data = txy_all[idx].to(self.device).detach()
            self.uvp_data = uvp_all[idx].to(self.device).detach()

        if self.use_time_marching and self.use_data_loss:
            warnings.warn(
                "When using both use_data_loss and use_time_marching, some time "
                "windows may contain no data points, or too few data points to make "
                "a data batch. Both cases currently result in an exception.",
                RuntimeWarning,
                stacklevel=2,
            )

    def _ref_uvp_at_time(self, time_moment: float):
        t_ref = self.t_ref
        u_ref = self.u_ref
        v_ref = self.v_ref
        p_ref = self.p_ref
        if time_moment <= t_ref[0]:
            return u_ref[:, 0], v_ref[:, 0], p_ref[:, 0]
        if time_moment >= t_ref[-1]:
            return u_ref[:, -1], v_ref[:, -1], p_ref[:, -1]

        idx = torch.searchsorted(t_ref, time_moment, right=True) - 1
        alpha = (time_moment - t_ref[idx]) / (t_ref[idx + 1] - t_ref[idx])

        u = (1.0 - alpha)*u_ref[:, idx] + alpha*u_ref[:, idx+1]
        v = (1.0 - alpha)*v_ref[:, idx] + alpha*v_ref[:, idx+1]
        p = (1.0 - alpha)*p_ref[:, idx] + alpha*p_ref[:, idx+1]

        return u, v, p

    def assemble_reference_txy_uvp(self, query_times: torch.Tensor):
        xy_ref = self.xy_ref
        t_ref = self.t_ref

        query_times = torch.as_tensor(query_times, dtype=t_ref.dtype, device=t_ref.device).flatten()

        txy_blocks = []
        uvp_blocks = []

        n_points = xy_ref.shape[0]

        for tm in query_times:
            u, v, p = self._ref_uvp_at_time(tm)

            t_column = torch.full((n_points, 1), tm, dtype=xy_ref.dtype, device=xy_ref.device)
            txy_block = torch.cat([t_column, xy_ref], dim=1)   # shape (n_points, 3)

            uvp_block = torch.stack([u, v, p], dim=1)             # shape (n_points, 3)

            txy_blocks.append(txy_block)
            uvp_blocks.append(uvp_block)

        txy = torch.cat(txy_blocks, dim=0)
        uvp = torch.cat(uvp_blocks, dim=0)

        return txy, uvp

    def sample_rectangle_minus_circle(self, n: int, oversample_factor: float = 1.2) -> torch.Tensor:
        x_min, x_max = self.xlims
        y_min, y_max = self.ylims
        t_min, t_max = self.tlims
        cx, cy = self.coords_obs
        r = self.d_obs / 2.0

        A_r = (x_max - x_min) * (y_max - y_min)
        A_c = torch.pi * r * r

        txy_total = []
        n_total = 0
        while n_total < n:
            N = math.ceil( oversample_factor * (n - n_total) * A_r / (A_r - A_c) )
            txy = sample_points_3D(
                [t_min, x_min, y_min, t_max, x_max, y_max], N, scheme="uniform", device=self.device
            )
            mask = (txy[:, 1] - cx)**2 + (txy[:, 2] - cy)**2 > r**2
            txy_total.append(txy[mask])
            n_total += len(txy_total[-1])

        return torch.cat(txy_total)[:n, :]

    def sample_points(self):
        x_min, x_max = self.xlims
        y_min, y_max = self.ylims
        t_min, t_max = self.tlims
        x_obs, y_obs = self.coords_obs
        d_obs = self.d_obs

        n_r, n_ic = self.n_r, self.n_ic
        n_bc_xmin, n_bc_xmax = self.n_bc_xmin, self.n_bc_xmax
        n_bc_ymin, n_bc_ymax = self.n_bc_ymin, self.n_bc_ymax
        n_bc_obs = self.n_bc_obs

        use_adaptive_sampling = self.use_adaptive_sampling

        if use_adaptive_sampling:
            factor = 5
        else:
            factor = 1

        # Domain
        txy_r = self.sample_rectangle_minus_circle(n_r*factor)
        if use_adaptive_sampling:
            (
                res_x, res_y, res_sigma_xx, res_sigma_xy, res_sigma_yy, res_p
            ) = self.compute_res_r(txy_r.detach().requires_grad_(True))
            res_r = torch.sqrt( res_x**2 + res_y**2 + res_sigma_xx**2 + res_sigma_xy**2 + res_sigma_yy**2 + res_p**2 )
            txy_r = weighted_subsample(txy_r, weights=res_r, k=n_r)

        # Initial conditions
        txy_ic = self.sample_rectangle_minus_circle(n_ic*factor)
        txy_ic[:, 0] = t_min
        if use_adaptive_sampling:
            res_u, res_v, res_p = self.compute_res_ic(txy_ic.detach().requires_grad_(True))
            res_ic = torch.sqrt( res_u**2 + res_v**2 + res_p**2 )
            txy_ic = weighted_subsample(txy_ic, weights=res_ic, k=n_ic)

        # Boundary conditions for rectangle boundaries
        txy_bc_xmin = sample_points_3D(
            [t_min, x_min, y_min, t_max, x_min, y_max],
            n_bc_xmin * factor, scheme="uniform", device=self.device
        )
        if use_adaptive_sampling:
            res_u, res_v = self.compute_res_inlet(txy_bc_xmin.detach().requires_grad_(True))
            res_inlet = torch.sqrt( res_u**2 + res_v**2 )
            txy_bc_xmin = weighted_subsample(txy_bc_xmin, weights=res_inlet, k=n_bc_xmin)

        txy_bc_xmax = sample_points_3D(
            [t_min, x_max, y_min, t_max, x_max, y_max],
            n_bc_xmax * factor, scheme="uniform", device=self.device
        )
        if use_adaptive_sampling:
            res_sigma_xx_out, res_sigma_xy_out = self.compute_res_outlet(txy_bc_xmax.detach().requires_grad_(True))
            txy_bc_xmax = weighted_subsample(txy_bc_xmax, weights=torch.sqrt( res_sigma_xx_out**2 + res_sigma_xy_out**2 ), k=n_bc_xmax)

        txy_bc_ymin = sample_points_3D(
            [t_min, x_min, y_min, t_max, x_max, y_min],
            n_bc_ymin * factor, scheme="uniform", device=self.device
        )
        txy_bc_ymax = sample_points_3D(
            [t_min, x_min, y_max, t_max, x_max, y_max],
            n_bc_ymax * factor, scheme="uniform", device=self.device
        )
        # Boundary conditions for obstacle
        tmp = sample_points_3D(
            [t_min, d_obs/2.0, 0.0, t_max, d_obs/2.0, 2.0*torch.pi],
            n_bc_obs * factor, scheme="uniform", device=self.device
        )
        txy_bc_obs = torch.zeros_like(tmp)
        txy_bc_obs[:, 0] = tmp[:, 0]
        txy_bc_obs[:, 1] = x_obs + tmp[:, 1] * torch.cos(tmp[:, 2])
        txy_bc_obs[:, 2] = y_obs + tmp[:, 1] * torch.sin(tmp[:, 2])
        if use_adaptive_sampling:
            u_ymin, v_ymin, u_ymax, v_ymax, u_obs, v_obs = self.compute_res_walls(
                txy_bc_ymin.detach().requires_grad_(True),
                txy_bc_ymax.detach().requires_grad_(True),
                txy_bc_obs.detach().requires_grad_(True)
            )

            res_ymin = torch.sqrt(u_ymin**2 + v_ymin**2)
            txy_bc_ymin = weighted_subsample(txy_bc_ymin, weights=res_ymin, k=n_bc_ymin)

            res_ymax = torch.sqrt(u_ymax**2 + v_ymax**2)
            txy_bc_ymax = weighted_subsample(txy_bc_ymax, weights=res_ymax, k=n_bc_ymax)

            res_obs = torch.sqrt(u_obs**2 + v_obs**2)
            txy_bc_obs = weighted_subsample(txy_bc_obs, weights=res_obs, k=n_bc_obs)

        txy_r.requires_grad_(True)
        txy_ic.requires_grad_(True)
        txy_bc_xmin.requires_grad_(True)
        txy_bc_xmax.requires_grad_(True)
        txy_bc_ymin.requires_grad_(True)
        txy_bc_ymax.requires_grad_(True)
        txy_bc_obs.requires_grad_(True)

        return txy_r, txy_ic, txy_bc_xmin, txy_bc_xmax, txy_bc_ymin, txy_bc_ymax, txy_bc_obs

    def sample_data_points(self):
        """
        Sample supervised data from global self.txy_data/self.uvp_data,
        but only inside the current self.tlims window.
        """
        t_min, t_max = self.tlims
        t = self.txy_data[:, 0]
        mask = (t > t_min) & (t <= t_max)

        txy_window = self.txy_data[mask]
        uvp_window = self.uvp_data[mask]

        if txy_window.shape[0] == 0:
            raise RuntimeError(
                f"No supervised data points found in current time window "
                f"[{t_min}, {t_max}]. Check self.t_ref/self.tlims scaling."
            )

        return txy_window.detach().requires_grad_(True), uvp_window.detach()

    def compute_res_r(self, txy, model=None):
        model = self.model if model is None else model
        out = model(txy)

        psi, p, sigma_xx, sigma_xy, sigma_yy = torch.split(out, 1, dim=1)
        u, v = get_uv_from_psi(txy, psi)
        du_dt, du_dx, du_dy = torch.split(compute_grad(u, txy), 1, dim=1)
        dv_dt, dv_dx, dv_dy = torch.split(compute_grad(v, txy), 1, dim=1)

        d_sigma_xx_dx = compute_grad(sigma_xx, txy)[:, 1:2]
        d_sigma_yy_dy = compute_grad(sigma_yy, txy)[:, 2:3]
        _, d_sigma_yx_dx, d_sigma_xy_dy = torch.split(compute_grad(sigma_xy, txy), 1, dim=1)

        Re_inv = self.Re_inv
        res_x = du_dt + u*du_dx + v*du_dy - d_sigma_xx_dx - d_sigma_xy_dy
        res_y = dv_dt + u*dv_dx + v*dv_dy - d_sigma_yx_dx - d_sigma_yy_dy
        res_sigma_xx = sigma_xx + p - 2.0*Re_inv*du_dx
        res_sigma_xy = sigma_xy - Re_inv*( du_dy + dv_dx )
        res_sigma_yy = sigma_yy + p - 2.0*Re_inv*dv_dy
        res_p = p + (sigma_xx + sigma_yy) / 2.0

        return res_x, res_y, res_sigma_xx, res_sigma_xy, res_sigma_yy, res_p

    def compute_initial_condition(self, txy):
        if self.pretrained_ic_model is None:
            if self.ic_source == "zero":
                zeros_tensor = torch.zeros((txy.shape[0], 1), dtype=txy.dtype, device=txy.device)
                u, v, p = zeros_tensor, zeros_tensor, zeros_tensor
            elif self.ic_source == "reference":
                xy_np = txy[:, 1:3].detach().cpu().numpy()
                u_np = self.ic_interp_u(xy_np)
                v_np = self.ic_interp_v(xy_np)
                p_np = self.ic_interp_p(xy_np)
                u = torch.as_tensor(u_np, dtype=txy.dtype, device=txy.device).reshape(-1, 1)
                v = torch.as_tensor(v_np, dtype=txy.dtype, device=txy.device).reshape(-1, 1)
                p = torch.as_tensor(p_np, dtype=txy.dtype, device=txy.device).reshape(-1, 1)
        else:
            txy_prev = txy.detach().clone().requires_grad_(True)
            out = self.pretrained_ic_model(txy_prev)
            psi = out[:, 0:1]
            u, v = get_uv_from_psi(txy_prev, psi)
            p = out[:, 1:2]
        return u.detach(), v.detach(), p.detach()

    def compute_res_ic(self, txy):
        out = self.model(txy)
        psi = out[:, 0:1]
        u, v = get_uv_from_psi(txy, psi)

        u_target, v_target, p_target = self.compute_initial_condition(txy)

        res_u = u - u_target
        res_v = v - v_target
        res_p = out[:, 1:2] - p_target

        return res_u, res_v, res_p

    def compute_res_walls(self, txy_ymin, txy_ymax, txy_obs):
        out_ymin = self.model(txy_ymin)
        u_ymin, v_ymin = get_uv_from_psi(txy_ymin, out_ymin[:, 0:1])

        out_ymax = self.model(txy_ymax)
        u_ymax, v_ymax = get_uv_from_psi(txy_ymax, out_ymax[:, 0:1])

        out_obs = self.model(txy_obs)
        u_obs, v_obs = get_uv_from_psi(txy_obs, out_obs[:, 0:1])

        return u_ymin, v_ymin, u_ymax, v_ymax, u_obs, v_obs

    def compute_res_inlet(self, txy):
        out = self.model(txy)
        u, v = get_uv_from_psi(txy, out[:, 0:1])

        res_u = (
            -self.u0 / self.h**2 * (txy[:, 2:3] - self.ylims[0]) * (txy[:, 2:3] - self.ylims[1]) *
            step_smoothed(txy[:, 0:1] * self.T_wave, x0=0.05) - u
        )
        res_v = v

        return res_u, res_v

    # (t, x, y) -> (psi, p, sigma_xx, sigma_xy, sigma_yy)
    def compute_res_outlet(self, txy):
        out = self.model(txy)
        sigma_xx = out[:, 2:3]
        sigma_xy = out[:, 3:4]
        return sigma_xx, sigma_xy

    def compute_res_data(self, txy, uvp):
        out = self.model(txy)
        psi = out[:, 0:1]
        u, v = get_uv_from_psi(txy, psi)

        res_u = u - uvp[:, 0:1]
        res_v = v - uvp[:, 1:2]
        #res_p = out[:, 1:2] - uvp[:, 2:3]

        return res_u, res_v#, res_p

    def compute_loss_terms(self, txy_r, txy_ic, txy_bc_xmin, txy_bc_xmax, txy_bc_ymin, txy_bc_ymax, txy_bc_obs,
            txy_data, uvp_data, lambdas, update_lambdas=False):
        model = self.model
        optimizer = self.optimizer
        use_causal_weighting_scheme = self.use_causal_weighting_scheme
        mse = lambda x: torch.mean(x**2)

        lambdas = dict(lambdas)
        losses = {name: 0.0 for name in ["total"] + list(lambdas.keys())}
        grad_losses = {name: 0.5 for name in list(lambdas.keys())}

        # In this context neural net acts like a map:
        # (t, x, y) -> (psi, p, sigma_xx, sigma_xy, sigma_yy)

        # Compute loss_r
        res_u_r, res_v_r, res_sigma_xx_r, res_sigma_xy_r, res_sigma_yy_r, res_p_r = self.compute_res_r(txy_r)
        losses["u_r"] = mse(res_u_r); losses["v_r"] = mse(res_v_r)
        losses["sigma_xx_r"] = mse(res_sigma_xx_r)
        losses["sigma_xy_r"] = mse(res_sigma_xy_r)
        losses["sigma_yy_r"] = mse(res_sigma_yy_r)
        losses["p_r"] = mse(res_p_r)
        if update_lambdas:
            for var in ["u_r", "v_r", "sigma_xx_r", "sigma_xy_r", "sigma_yy_r", "p_r"]:
                optimizer.zero_grad()
                losses[var].backward(retain_graph=True)
                grad_losses[var] = compute_grad_theta_norm(model)

        # loss_r = lambda_u_r*res_u_r + lambda_v_r*res_v_r + lambda_sigma_xx_r*res_sigma_xx_r +
        #     + lambda_sigma_xy_r*res_sigma_xy_r + lambda_sigma_yy_r*res_sigma_yy_r + lambda_p_r*res_p_r
        if use_causal_weighting_scheme:
            edges = self.causal_edges    # границы временнЫх интервалов
            bin_idx = torch.bucketize(txy_r[:, 0].detach(), edges[1:-1])      # к какому временнОму интервалу принадлежит каждая точка

            res_u_r_sq = res_u_r.squeeze(-1)**2
            res_v_r_sq = res_v_r.squeeze(-1)**2
            res_sigma_xx_r_sq = res_sigma_xx_r.squeeze(-1)**2
            res_sigma_xy_r_sq = res_sigma_xy_r.squeeze(-1)**2
            res_sigma_yy_r_sq = res_sigma_yy_r.squeeze(-1)**2
            res_p_r_sq = res_p_r.squeeze(-1)**2

            L_u_r_sum = torch.zeros(self.causal_m, dtype=res_u_r_sq.dtype, device=self.device)
            L_v_r_sum = torch.zeros(self.causal_m, dtype=res_v_r_sq.dtype, device=self.device)
            L_sigma_xx_r_sum = torch.zeros(self.causal_m, dtype=res_sigma_xx_r_sq.dtype, device=self.device)
            L_sigma_xy_r_sum = torch.zeros(self.causal_m, dtype=res_sigma_xy_r_sq.dtype, device=self.device)
            L_sigma_yy_r_sum = torch.zeros(self.causal_m, dtype=res_sigma_yy_r_sq.dtype, device=self.device)
            L_p_r_sum = torch.zeros(self.causal_m, dtype=res_p_r_sq.dtype, device=self.device)

            L_u_r_sum.index_add_(0, bin_idx, res_u_r_sq)    # после этого в L_u_r_sum будет сумма res_u_r**2 на каждый временнОй интервал
            L_v_r_sum.index_add_(0, bin_idx, res_v_r_sq)
            L_sigma_xx_r_sum.index_add_(0, bin_idx, res_sigma_xx_r_sq)
            L_sigma_xy_r_sum.index_add_(0, bin_idx, res_sigma_xy_r_sq)
            L_sigma_yy_r_sum.index_add_(0, bin_idx, res_sigma_yy_r_sq)
            L_p_r_sum.index_add_(0, bin_idx, res_p_r_sq)

            counts = torch.bincount(bin_idx, minlength=self.causal_m).clamp_min(1).to(res_u_r_sq.dtype)    # сколько точек в каждом временнОм интервале
            L_u_r_t = L_u_r_sum / counts    # после этого в L_t будет loss на каждый временнОй интервал
            L_v_r_t = L_v_r_sum / counts
            L_sigma_xx_r_t = L_sigma_xx_r_sum / counts
            L_sigma_xy_r_t = L_sigma_xy_r_sum / counts
            L_sigma_yy_r_t = L_sigma_yy_r_sum / counts
            L_p_r_t = L_p_r_sum / counts

            L_t = lambdas["u_r"]*L_u_r_t + lambdas["v_r"]*L_v_r_t + lambdas["sigma_xx_r"]*L_sigma_xx_r_t + \
                lambdas["sigma_xy_r"]*L_sigma_xy_r_t + lambdas["sigma_yy_r"]*L_sigma_yy_r_t + lambdas["p_r"]*L_p_r_t

            prefix = torch.cumsum(L_t.detach(), dim=0) - L_t.detach()
            W = torch.exp(-self.causal_eps * prefix)
            loss_r = (W * L_t).mean()
        else:
            loss_r = lambdas["u_r"]*losses["u_r"] + lambdas["v_r"]*losses["v_r"] + \
                lambdas["sigma_xx_r"]*losses["sigma_xx_r"] + lambdas["sigma_xy_r"]*losses["sigma_xy_r"] + \
                lambdas["sigma_yy_r"]*losses["sigma_yy_r"] + lambdas["p_r"]*losses["p_r"]

        # Compute loss_ic
        res_u_ic, res_v_ic, res_p_ic = self.compute_res_ic(txy_ic)
        losses["u_ic"] = mse(res_u_ic); losses["v_ic"] = mse(res_v_ic); losses["p_ic"] = mse(res_p_ic)
        if update_lambdas:
            for var in ["u_ic", "v_ic", "p_ic"]:
                optimizer.zero_grad()
                losses[var].backward(retain_graph=True)
                grad_losses[var] = compute_grad_theta_norm(model)

        # Compute loss_walls
        res_u_ymin, res_v_ymin, res_u_ymax, res_v_ymax, res_u_obs, res_v_obs = self.compute_res_walls(txy_bc_ymin, txy_bc_ymax, txy_bc_obs)
        losses["u_walls"] = mse(res_u_ymin) + mse(res_u_ymax) + mse(res_u_obs)
        losses["v_walls"] = mse(res_v_ymin) + mse(res_v_ymax) + mse(res_v_obs)
        if update_lambdas:
            for var in ["u_walls", "v_walls"]:
                optimizer.zero_grad()
                losses[var].backward(retain_graph=True)
                grad_losses[var] = compute_grad_theta_norm(model)

        # Compute loss_inlet
        res_u_inlet, res_v_inlet = self.compute_res_inlet(txy_bc_xmin)
        losses["u_in"] = mse(res_u_inlet); losses["v_in"] = mse(res_v_inlet)
        if update_lambdas:
            for var in ["u_in", "v_in"]:
                optimizer.zero_grad()
                losses[var].backward(retain_graph=True)
                grad_losses[var] = compute_grad_theta_norm(model)

        # Compute loss_outlet
        res_sigma_xx_outlet, res_sigma_xy_outlet = self.compute_res_outlet(txy_bc_xmax)
        losses["sigma_xx_out"] = mse(res_sigma_xx_outlet)
        losses["sigma_xy_out"] = mse(res_sigma_xy_outlet)
        if update_lambdas:
            for var in ["sigma_xx_out", "sigma_xy_out"]:
                optimizer.zero_grad()
                losses[var].backward(retain_graph=True)
                grad_losses[var] = compute_grad_theta_norm(model)

        # Compute loss_data
        if self.use_data_loss:
            #res_u_data, res_v_data, res_p_data = self.compute_res_data(txy_data, uvp_data)
            res_u_data, res_v_data = self.compute_res_data(txy_data, uvp_data)
            losses["u_data"] = mse(res_u_data)
            losses["v_data"] = mse(res_v_data)
            #losses["p_data"] = mse(res_p_data)
            if update_lambdas:
                #for var in ["u_data", "v_data", "p_data"]:
                for var in ["u_data", "v_data"]:
                    optimizer.zero_grad()
                    losses[var].backward(retain_graph=True)
                    grad_losses[var] = compute_grad_theta_norm(model)

        # Compute new lambdas if needed
        if update_lambdas:
            tmp = sum(grad_losses.values())
            alpha = self.grad_norm_weighting_alpha
            for var in list(lambdas.keys()):
                lambdas[var] = alpha*lambdas[var] + (1.0-alpha)*tmp/max(float(grad_losses[var]), 1e-12)

        weighted_loss_names = [name for name in lambdas if not name.endswith("_r")]
        losses["total"] = loss_r + sum(losses[k] * lambdas[k] for k in weighted_loss_names)

        return (losses, lambdas,
            W if use_causal_weighting_scheme else None, L_t if use_causal_weighting_scheme else None)

    def _prepare_reference_comparison_data(self, t_min, t_max):
        txy, uvp_exact_arr = self.assemble_reference_txy_uvp(torch.linspace(t_min, t_max, 50))
        txy_rg = txy.clone().detach().requires_grad_(True)
        uvp_exact_arr_l2 = torch.norm(uvp_exact_arr, p=2).item()
        u_exact_arr_l2 = torch.norm(uvp_exact_arr[:, 0], p=2).item()
        v_exact_arr_l2 = torch.norm(uvp_exact_arr[:, 1], p=2).item()
        p_exact_arr_l2 = torch.norm(uvp_exact_arr[:, 2], p=2).item()
        uvp_exact_arr = uvp_exact_arr.cpu()
        return txy_rg, uvp_exact_arr, uvp_exact_arr_l2, u_exact_arr_l2, v_exact_arr_l2, p_exact_arr_l2

    def train(self):
        n_epochs = self.cfg.n_epochs
        n_batches = self.cfg.n_batches
        batch_size_r, batch_size_ic = self.cfg.batch_size_r, self.cfg.batch_size_ic
        batch_size_bc_xmin, batch_size_bc_xmax = self.cfg.batch_size_bc_xmin, self.cfg.batch_size_bc_xmax
        batch_size_bc_ymin, batch_size_bc_ymax = self.cfg.batch_size_bc_ymin, self.cfg.batch_size_bc_ymax
        batch_size_bc_obs = self.cfg.batch_size_bc_obs

        use_grad_norm_weighting = self.cfg.use_grad_norm_weighting
        grad_norm_weighting_freq = self.cfg.grad_norm_weighting_freq
        use_causal = self.use_causal_weighting_scheme
        use_time_marching = self.use_time_marching
        self.pretrained_ic_model = self._load_pretrained_ic_model()
        lambdas = self.cfg.lambdas
        use_static_training_set = self.use_static_training_set
        W_logging_freq = self.cfg.W_logging_freq
        logging_freq = self.cfg.logging_freq
        t_min, t_max = self.tlims
        render_video = self.cfg.render_video
        model = self.model
        use_checkpointing = self.cfg.use_checkpointing
        checkpointing_freq = self.cfg.checkpointing_freq
        results_folder = self.results_folder
        n_iters = n_epochs * n_batches
        resampling_frequency = self.cfg.resampling_frequency

        use_temporal_logging = self.use_temporal_logging
        temporal_logging_freq = self.temporal_logging_freq
        temporal_logging_nbins = self.temporal_logging_nbins

        use_data_loss = self.use_data_loss

        # Для логирования
        n_log = len(range(0, n_iters, logging_freq))
        n_wlog = n_iters // W_logging_freq
        n_tmp_log = n_iters // temporal_logging_freq
        logged_layers = self.logged_layers
        n_logged_layers = len(logged_layers)
        lambda_names = list(self.cfg.lambdas.keys())
        loss_names = ["total"] + lambda_names
        metrics = {
            "step": torch.arange(0, n_iters, logging_freq),
            "losses": {name: torch.zeros(n_log) for name in loss_names},

            "err_l2": torch.zeros(n_log),
            "rel_err_l2": torch.zeros(n_log),
            "err_u_l2": torch.zeros(n_log),
            "rel_err_u_l2": torch.zeros(n_log),
            "err_v_l2": torch.zeros(n_log),
            "rel_err_v_l2": torch.zeros(n_log),
            "err_p_l2": torch.zeros(n_log),
            "rel_err_p_l2": torch.zeros(n_log),

            "lambdas": {name: torch.zeros(n_log) for name in lambda_names},

            # Per-layer weight and biases mean squares
            "per_layer_weight_ms": torch.zeros(n_log, n_logged_layers),
            "per_layer_bias_ms": torch.zeros(n_log, n_logged_layers),
            "per_layer_weight_grad_ms": torch.zeros(n_log, n_logged_layers),
            "per_layer_bias_grad_ms": torch.zeros(n_log, n_logged_layers),

            "lr": torch.zeros(n_log),

            "W_step": torch.arange(0, n_iters, W_logging_freq),
            "W": [None] * n_wlog,
            "w_min": torch.zeros(n_log),
            "L_t": [None] * n_wlog,

            "temporal_logging_step": torch.arange(0, n_iters, temporal_logging_freq),
            "temporal_loss_r": [None] * n_tmp_log,
            "temporal_error": [None] * n_tmp_log
        }

        # Video
        video_path = results_folder / ("training_weights_animation" + ".mp4")
        video_render_freq = self.cfg.video_render_freq

        ### ДЛЯ АНИМАЦИИ ПРОЦЕССА ОБУЧЕНИЯ
        if render_video:
            writer = anim.FFMpegWriter(
                fps=self.cfg.video_fps,
                codec='libx264',
                extra_args=['-pix_fmt', 'yuv420p', '-preset', 'ultrafast', "-threads", "0"]
            )
            w, h = plt.rcParams['figure.figsize']
            w *= 0.6
            h *= 0.6

            fig, axes = plt.subplots(
                4, n_logged_layers,
                figsize=(n_logged_layers*w, 4*h), squeeze=False, constrained_layout=False)

            row_labels = ["weights", "biases", "weights grad", "biases grad"]
            for i, label in enumerate(row_labels):
                axes[i, 0].set_ylabel(label, rotation=90, labelpad=20)
            for j, name in enumerate(self.logged_layer_names):
                axes[0, j].set_title(name, pad=10)

            n_bins = 100
            st = np.empty_like(axes)
            for i in range(axes.shape[0]):
                for j in range(axes.shape[1]):
                    if i == 0 or i == 2:
                        data = logged_layers[j].weight
                    elif i == 1 or i == 3:
                        data = logged_layers[j].bias
                    counts, edges = torch.histogram(torch.rand_like(data).cpu(), bins=n_bins)
                    st[i, j] = axes[i, j].stairs(counts.cpu(), edges.cpu(), fill=True, baseline=0, color='C0')

        pbar = tqdm(total=n_iters, desc="Training model")

        def train_one_window(step_start: int, step_end: int, t_min: float, t_max: float, lambdas: dict):
            txy_rg, uvp_exact_arr, uvp_exact_arr_l2, u_exact_arr_l2, v_exact_arr_l2, p_exact_arr_l2 = (
                self._prepare_reference_comparison_data(t_min, t_max)
            )
            self.tlims = [t_min, t_max]

            self.causal_edges = torch.linspace(t_min, t_max, self.causal_m + 1, device=self.device)

            # Data points
            if self.use_data_loss:
                txy_data, uvp_data = self.sample_data_points()
                batch_size_data = math.ceil(txy_data.shape[0] / n_batches)
                if batch_size_data < 1:
                    raise ValueError(f"batch_size_data turned out to be less than 1 ({batch_size_data:.1f}). ")

            causal_eps_schedule = self.causal_eps_schedule
            causal_eps_id = 0
            self.causal_eps = causal_eps_schedule[0]

            for step in range(step_start, step_end):
                i_batch = (step - step_start) % n_batches    # batch index inside the current epoch
                if step == step_start or (not use_static_training_set and (step - step_start) % resampling_frequency == 0):
                    txy_r, txy_ic, txy_bc_xmin, txy_bc_xmax, txy_bc_ymin, txy_bc_ymax, txy_bc_obs = self.sample_points()
                if use_static_training_set:    # re-leaf tensors
                    txy_r = txy_r.detach().requires_grad_(True)
                    txy_ic = txy_ic.detach().requires_grad_(True)
                    txy_bc_xmin = txy_bc_xmin.detach().requires_grad_(True)
                    txy_bc_xmax = txy_bc_xmax.detach().requires_grad_(True)
                    txy_bc_ymin = txy_bc_ymin.detach().requires_grad_(True)
                    txy_bc_ymax = txy_bc_ymax.detach().requires_grad_(True)
                    txy_bc_obs = txy_bc_obs.detach().requires_grad_(True)
                    txy_data = txy_data.detach().requires_grad_(True) if use_data_loss else None
                losses, lambdas, W, L_t = self.compute_loss_terms(
                    txy_r[i_batch*batch_size_r:(i_batch+1)*batch_size_r],
                    txy_ic[i_batch*batch_size_ic:(i_batch+1)*batch_size_ic],
                    txy_bc_xmin[i_batch*(batch_size_bc_xmin):(i_batch+1)*(batch_size_bc_xmin)],
                    txy_bc_xmax[i_batch*(batch_size_bc_xmax):(i_batch+1)*(batch_size_bc_xmax)],
                    txy_bc_ymin[i_batch*(batch_size_bc_ymin):(i_batch+1)*(batch_size_bc_ymin)],
                    txy_bc_ymax[i_batch*(batch_size_bc_ymax):(i_batch+1)*(batch_size_bc_ymax)],
                    txy_bc_obs[i_batch*batch_size_bc_obs:(i_batch+1)*batch_size_bc_obs],
                    txy_data[i_batch*batch_size_data:(i_batch+1)*batch_size_data] if use_data_loss else None,
                    uvp_data[i_batch*batch_size_data:(i_batch+1)*batch_size_data] if use_data_loss else None,
                    lambdas=lambdas,
                    update_lambdas = use_grad_norm_weighting and step % grad_norm_weighting_freq == 0
                )

                self.optimizer.zero_grad()
                losses["total"].backward()

                if step % logging_freq == 0:
                    i = step // logging_freq
                    log_per_layer_ms(i, logged_layers,
                        metrics["per_layer_weight_ms"],
                        metrics["per_layer_bias_ms"],
                        metrics["per_layer_weight_grad_ms"],
                        metrics["per_layer_bias_grad_ms"]
                    )

                causal_w_min = W.detach().min().cpu().item() if use_causal else 0.0
                if use_causal and causal_w_min > 0.95 and causal_eps_id < len(causal_eps_schedule) - 1:
                    causal_eps_id += 1
                    self.causal_eps = causal_eps_schedule[causal_eps_id]
                if use_causal and step % W_logging_freq == 0:
                    i = step // W_logging_freq
                    metrics["W"][i] = W.detach().cpu().numpy()
                    metrics["L_t"][i] = L_t.detach().cpu().numpy()

                if step % logging_freq == 0:
                    i = step // logging_freq
                    for var in loss_names:
                        metrics["losses"][var][i] = losses[var].item()
                    model_output = model(txy_rg)
                    psi = model_output[:, 0:1]
                    u, v = get_uv_from_psi(txy_rg, psi)
                    p = model_output[:, 1:2]
                    uvp_model = torch.cat([u, v, p], dim=1).detach().cpu()
                    metrics["err_l2"][i] = torch.norm(uvp_exact_arr - uvp_model, p=2).item()
                    metrics["rel_err_l2"][i] = metrics["err_l2"][i] / uvp_exact_arr_l2 * 100

                    metrics["err_u_l2"][i] = torch.norm(uvp_exact_arr[:, 0] - uvp_model[:, 0], p=2).item()
                    metrics["rel_err_u_l2"][i] = metrics["err_u_l2"][i] / u_exact_arr_l2 * 100
                    metrics["err_v_l2"][i] = torch.norm(uvp_exact_arr[:, 1] - uvp_model[:, 1], p=2).item()
                    metrics["rel_err_v_l2"][i] = metrics["err_v_l2"][i] / v_exact_arr_l2 * 100
                    metrics["err_p_l2"][i] = torch.norm(uvp_exact_arr[:, 2] - uvp_model[:, 2], p=2).item()
                    metrics["rel_err_p_l2"][i] = metrics["err_p_l2"][i] / p_exact_arr_l2 * 100
                    metrics["w_min"][i] = causal_w_min

                    for var in lambda_names:
                        metrics["lambdas"][var][i] = lambdas[var]

                    metrics["lr"][i] = self.optimizer.param_groups[0]["lr"]

                    pbar.set_postfix({
                        'loss': metrics["losses"]["total"][i].item(),
                        "rel_err_u_l2": metrics["rel_err_u_l2"][i].item(),
                        "rel_err_v_l2": metrics["rel_err_v_l2"][i].item(),
                        "rel_err_p_l2": metrics["rel_err_p_l2"][i].item(),
                        "causal_w_min": causal_w_min,
                        "causal_eps": self.causal_eps
                    })

                # Рендер кадра анимации
                if render_video and step % video_render_freq == 0:
                    fig.suptitle(f"Grad step: {step:6d}")

                    for i in range(axes.shape[0]):
                        for j in range(axes.shape[1]):
                            if i == 0:
                                data = logged_layers[j].weight
                            elif i == 1:
                                data = logged_layers[j].bias
                            elif i == 2:
                                data = logged_layers[j].weight.grad
                            elif i == 3:
                                data = logged_layers[j].bias.grad
                            if data is None:
                                continue
                            counts, edges = torch.histogram(data.detach().cpu(), bins=n_bins)
                            st[i, j].set_data(values=counts.cpu(), edges=edges.cpu())
                            xlims_new = [edges.min().item(), edges.max().item()]
                            ylims_new = [counts.min().item(), counts.max().item()]
                            update_axis_limits_hysteresis(axes[i, j], xlims_new, ylims_new)

                    writer.grab_frame()

                if use_checkpointing and (step + 1) % checkpointing_freq == 0:
                    # save model
                    model_path = results_folder / "model" / ("model_epoch=" + str(step//n_batches + 1) +
                        "_batch=" + str(step % n_batches + 1) + ".pth")
                    MultilayerPerceptronWithFFE.save(model, model_path)

                    # save metrics
                    metrics_path = results_folder / "metrics" / ("metrics_epoch=" + str(step//n_batches + 1) +
                        "_batch=" + str(step % n_batches + 1) + ".pth")
                    torch.save(metrics, metrics_path)

                self.optimizer.step()
                if self.scheduler is not None:
                    self.scheduler.step()
                pbar.update(1)

            return lambdas

        full_tlims = self.tlims[:]

        # Mark arrays for time marching
        if use_time_marching:
            time_borders = torch.linspace(full_tlims[0], full_tlims[1], self.n_time_windows + 1).tolist()
            steps_per_window = n_iters // self.n_time_windows
            windows = [
                (
                    window_idx,
                    window_idx * steps_per_window,
                    n_iters if window_idx == self.n_time_windows - 1 else (window_idx + 1) * steps_per_window,
                    time_borders[window_idx],
                    time_borders[window_idx + 1],
                )
                for window_idx in range(self.n_time_windows)
            ]
        else:
            windows = [(0, 0, n_iters, full_tlims[0], full_tlims[1])]

        with writer.saving(fig, video_path, dpi=self.cfg.video_dpi) if render_video else nullcontext():
            for window_idx, step_start, step_end, t_min, t_max in windows:
                pbar.set_description(f"Window {window_idx + 1}/{len(windows)}")

                self.tlims = [t_min, t_max]

                if use_time_marching and window_idx > 0:
                    self.pretrained_ic_model = copy.deepcopy(model).eval()
                    initialize_weights(model, model.init_scheme)
                    for p in self.pretrained_ic_model.parameters():
                        p.requires_grad_(False)
                    lambdas = dict(self.cfg.lambdas)

                self.optimizer = self.optimizer_factory(model)
                self.scheduler = self.scheduler_factory(self.optimizer)

                lambdas = train_one_window(
                    step_start=step_start, step_end=step_end,
                    t_min=t_min, t_max=t_max,
                    lambdas=lambdas
                )

                if use_time_marching:
                    MultilayerPerceptronWithFFE.save(model, results_folder / "model" / f"model_window={window_idx}.pth")

        pbar.close()

        if use_time_marching:
            self.tlims = [float(full_tlims[0]), float(full_tlims[-1])]
        self.pretrained_ic_model = self._load_pretrained_ic_model()

        return metrics

    def model_at_time(self, t: float | torch.Tensor, device: str | torch.device = "cpu") -> nn.Module:
        """
        Return the sub-model responsible for the time window containing ``t``.

        When time marching is disabled, the single live model ``self.model`` is
        returned unchanged. When time marching is enabled, the training run has
        saved one checkpoint per window (``model_window={w}.pth``); this method
        resolves which window ``t`` falls into and lazily loads the corresponding
        checkpoint from disk.

        To avoid holding every window in memory at once, only the most recently
        requested ``(window, device)`` pair is cached. A checkpoint is loaded from
        disk only when the requested window or device differs from the cached one;
        repeated calls that resolve to the same window reuse the cached model.
        """
        if not self.use_time_marching:
            return self.model

        device = torch.device(device)

        borders = torch.linspace(self.tlims[0], self.tlims[1], self.n_time_windows + 1)
        w = int(torch.searchsorted(borders, torch.as_tensor(float(t)), right=True).item()) - 1
        w = max(0, min(w, self.n_time_windows - 1))

        if getattr(self, "_cached_window", None) != w or getattr(self, "_cached_device", None) != device:
            path = self.results_folder / "model" / f"model_window={w}.pth"
            self._cached_model = MultilayerPerceptronWithFFE.load(path, device=device).eval()
            self._cached_window = w
            self._cached_device = device
        return self._cached_model

    def render_metrics_plots(self, metrics: dict, show=True, save_to_disk: bool = False, fmt: str ="png"):
        plot_path = self.results_folder / "plots"
        if save_to_disk:
            plot_path.mkdir(parents=True, exist_ok=True)

        w, h = plt.rcParams['figure.figsize']

        def show_plot(fig, show):
            if not show:
                plt.close(fig)

        def save_fig(fig, save_path: Path, dpi: int = 300):
            if save_to_disk:
                fig.savefig(save_path, dpi=dpi)

        def linestyle_for_loss(name: str):
            if name.endswith("_r"):
                return "-"
            if name.endswith("_ic"):
                return "--"
            if name.endswith("_walls"):
                return "-."
            if name.endswith("_in"):
                return ":"
            if name.endswith("_out"):
                return (0, (5, 2))   # custom dashed style
            return "-"               # fallback, e.g. "total"

        # Losses
        fig, ax = plt.subplots()
        ax.set(title="Losses", xlabel="grad step")
        for var in list(metrics["losses"].keys()):
            ax.semilogy(metrics["step"], metrics["losses"][var], label=var, linestyle=linestyle_for_loss(var))
        ax.legend()
        show_plot(fig, show)
        save_fig(fig, plot_path / ("losses" + "." + fmt))
        # fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6)) = plt.subplots(
        #     3, 2, figsize=(2*w, 3*h), constrained_layout=True)
        # ax1.semilogy(metrics["step"], metrics["loss"], label="loss", color='C0')
        # ax2.semilogy(metrics["step"], metrics["loss_r"], label="loss_r", color='C1')
        # ax3.semilogy(metrics["step"], metrics["loss_ic"], label="loss_ic", color='C2')
        # ax4.semilogy(metrics["step"], metrics["loss_walls"], label="loss_walls", color='C3')
        # ax5.semilogy(metrics["step"], metrics["loss_inlet"], label="loss_inlet", color='C4')
        # ax6.semilogy(metrics["step"], metrics["loss_outlet"], label="loss_outlet", color='C5')
        # for ax in [ax1, ax2, ax3, ax4, ax5, ax6]:
        #     ax.set(xlabel="grad step")
        #     ax.legend()
        # show_plot(fig, show)
        # save_fig(fig, plot_path / ("losses_separate" + "." + fmt))

        # Error
        fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8)) = plt.subplots(
            4, 2, figsize=(2*w, 4*h), constrained_layout=True)
        ax1.plot(metrics["step"], metrics["err_l2"], label="validate")
        ax1.set(title="Error L2 norm", xlabel="grad step")
        ax2.plot(metrics["step"], metrics["rel_err_l2"])
        ax2.set(title="Relative error L2 norm", xlabel="grad step", ylabel="%")
        ax3.plot(metrics["step"], metrics["err_u_l2"])
        ax3.set(title="Error u L2 norm", xlabel="grad step")
        ax4.plot(metrics["step"], metrics["rel_err_u_l2"])
        ax4.set(title="Relative error u L2 norm", xlabel="grad step")
        ax5.plot(metrics["step"], metrics["err_v_l2"])
        ax5.set(title="Error v L2 norm", xlabel="grad step")
        ax6.plot(metrics["step"], metrics["rel_err_v_l2"])
        ax6.set(title="Relative error v L2 norm", xlabel="grad step")
        ax7.plot(metrics["step"], metrics["err_p_l2"])
        ax7.set(title="Error p L2 norm", xlabel="grad step")
        ax8.plot(metrics["step"], metrics["rel_err_p_l2"])
        ax8.set(title="Relative error p L2 norm", xlabel="grad step")
        show_plot(fig, show)
        save_fig(fig, plot_path / ("error" + "." + fmt))

        # Lambdas
        if self.cfg.use_grad_norm_weighting:
            fig, ax = plt.subplots()
            ax.set(title="Lambdas", xlabel="grad step")
            for var in list(metrics["lambdas"].keys()):
                ax.semilogy(metrics["step"], metrics["lambdas"][var], label=var, linestyle=linestyle_for_loss(var))
            ax.legend()
            show_plot(fig, show)
            save_fig(fig, plot_path / ("lambdas" + "." + fmt))

        # Per-layer weight and biases mean squares
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(2*w, 2*h), constrained_layout=True)
        layer_names = self.logged_layer_names
        for j in range(len(self.logged_layers)):
            ax1.plot(metrics["step"], metrics["per_layer_weight_ms"][:, j], label=layer_names[j])
            ax2.plot(metrics["step"], metrics["per_layer_bias_ms"][:, j], label=layer_names[j])
            ax3.semilogy(metrics["step"], metrics["per_layer_weight_grad_ms"][:, j], label=layer_names[j])
            ax4.semilogy(metrics["step"], metrics["per_layer_bias_grad_ms"][:, j], label=layer_names[j])
        ax1.set(title="Weight mean square by layer")
        ax2.set(title="Bias mean square by layer")
        ax3.set(title="Weight gradient mean square by layer")
        ax4.set(title="Bias gradient mean square by layer")
        for ax in [ax1, ax2, ax3, ax4]:
            ax.set(xlabel="grad step")
            ax.legend()
        show_plot(fig, show)
        save_fig(fig, plot_path / ("weights_mean_squares" + "." + fmt))

        # Temporal weights
        if self.cfg.use_causal_weighting_scheme:
            W = np.stack(metrics["W"])
            t = np.arange(W.shape[1])
            fig, ax = plt.subplots()
            pm = ax.pcolormesh(t, metrics["W_step"], W, shading='auto', cmap='binary')
            cbar = fig.colorbar(pm, ax=ax)
            ax.set(title="Temporal weights $w_i$ over training", xlabel="$i$", ylabel="gradient step")
            ax.grid(False)
            show_plot(fig, show)
            save_fig(fig, plot_path / ("temporal_weights" + "." + fmt))

        # L_t
        if self.cfg.use_causal_weighting_scheme:
            L_t = np.stack(metrics["L_t"])
            t = np.arange(L_t.shape[1])
            fig, ax = plt.subplots()
            pm = ax.pcolormesh(t, metrics["W_step"], L_t, shading='auto', cmap='binary',
                norm=colors.LogNorm())
            cbar = fig.colorbar(pm, ax=ax)
            ax.set(title="$\\mathcal{L}_{r}^{i}$ for gradient steps", xlabel="$i$", ylabel="grad step")
            ax.grid(False)
            show_plot(fig, show)
            save_fig(fig, plot_path / ("l_t" + "." + fmt))

        if self.cfg.use_causal_weighting_scheme:
            fig, ax = plt.subplots()
            ax.plot(metrics["step"], metrics["w_min"])
            ax.set(title="min(W)", xlabel="grad step")
            show_plot(fig, show)
            save_fig(fig, plot_path / ("w_min" + "." + fmt))

        # Learning rate
        fig, ax = plt.subplots()
        ax.set(title="Learning rate", xlabel="grad step")
        ax.semilogy(metrics["step"], metrics["lr"])
        show_plot(fig, show)
        save_fig(fig, plot_path / ("learning_rate" + "." + fmt))

        # Temporal error
        if self.cfg.use_temporal_logging:
            error_t = np.stack(metrics["temporal_error"])
            t = np.arange(error_t.shape[1])
            fig, ax = plt.subplots()
            pm = ax.pcolormesh(t, metrics["temporal_logging_step"], error_t[:, :, 0], shading='auto', cmap='binary')
            cbar = fig.colorbar(pm, ax=ax)
            ax.set(title="$\\mathcal{error}_{r}^{i}$ for gradient steps", xlabel="$i$", ylabel="grad step")
            ax.grid(False)
            show_plot(fig, show)
            save_fig(fig, plot_path / ("error_t" + "." + fmt))

        # Lift and drag coefficients
        n_t = 100
        n_theta = 500
        xc, yc = self.coords_obs
        r = self.d_obs/2.0
        t_arr = torch.linspace(self.tlims[0], self.tlims[1], n_t)
        model = self.model.to("cpu")
        theta_arr = torch.linspace(0.0, 2.0*torch.pi, n_theta)
        C_L_arr = torch.zeros_like(t_arr)
        C_D_arr = torch.zeros_like(t_arr)
        with torch.no_grad():
            for i, t in enumerate(t_arr):
                txy = torch.stack(
                    [torch.full_like(theta_arr, t), xc + r*torch.cos(theta_arr), yc + r*torch.sin(theta_arr)],
                dim=1)
                if self.use_time_marching:
                    model = self.model_at_time(t)
                out = model(txy); sigma_xx = out[:, 2]; sigma_xy = out[:, 3]; sigma_yy = out[:, 4]
                F_L = r * torch.trapezoid(sigma_xy*torch.cos(theta_arr) + sigma_yy*torch.sin(theta_arr), theta_arr) * self.cfg.Sigma_wave
                F_D = r * torch.trapezoid(sigma_xx*torch.cos(theta_arr) + sigma_xy*torch.sin(theta_arr), theta_arr) * self.cfg.Sigma_wave
                C_L_arr[i] = F_L/(self.rho*self.U_wave**2*r)
                C_D_arr[i] = F_D/(self.rho*self.U_wave**2*r)
        tlims = (self.tlims[0]*self.T_wave, self.tlims[1]*self.T_wave)
        t_ref = self.t_ref.cpu() * self.T_wave
        cl_ref = self.cl_ref.cpu()
        cd_ref = self.cd_ref.cpu()
        fig, ax = plt.subplots()
        ax.plot(t_arr.detach()*self.T_wave, C_L_arr.detach(), label="model")
        ax.plot(t_ref, cl_ref, label="reference")
        ax.set(title="Lift coefficient", xlabel="t, sec", ylabel="C_L", xlim=tlims)
        ax.legend()
        show_plot(fig, show)
        save_fig(fig, plot_path / ("C_L" + "." + fmt))
        fig, ax = plt.subplots()
        ax.plot(t_arr.detach()*self.T_wave, C_D_arr.detach(), label="model")
        ax.plot(t_ref, cd_ref, label="reference")
        ax.set(title="Drag coefficient", xlabel="t, sec", ylabel="C_D", xlim=tlims)
        ax.legend()
        show_plot(fig, show)
        save_fig(fig, plot_path / ("C_D" + "." + fmt))
        self.model.to(self.device)

    def render_model_vs_reference_video(self, video_fps: int = 30, video_length: float = 3.0, video_dpi: int = 150):
        t_min, t_max = self.tlims
        x_min, x_max = self.xlims
        y_min, y_max = self.ylims

        x_obs, y_obs = self.coords_obs
        d_obs = self.d_obs

        L_wave = self.cfg.L_wave
        T_wave = self.cfg.T_wave
        U_wave = self.cfg.U_wave
        P_wave = self.cfg.P_wave

        model = self.model.to("cpu")
        t_ref = self.t_ref.cpu()
        xy_ref = self.xy_ref.cpu()
        u_ref = self.u_ref.cpu()
        v_ref = self.v_ref.cpu()
        p_ref = self.p_ref.cpu()

        def model_txy_to_uvp(model, txy):
            out = model(txy)
            psi, p = out[:, 0:1], out[:, 1]
            u, v = get_uv_from_psi(txy, psi, create_graph=False)
            return u.squeeze(), v.squeeze(), p

        n = 10000
        nx = math.ceil( math.sqrt(n * (x_max - x_min) / (y_max - y_min)) )
        ny = math.ceil(n / nx)

        # Get min and max values for u, v, p
        txy = torch.cat(
            [
                torch.ones((xy_ref.shape[0], 1), dtype=xy_ref.dtype, device=xy_ref.device),
                xy_ref
            ],
            dim=1
        ).requires_grad_(True)

        n_frames = math.ceil(video_fps * video_length)
        t_step = (t_max - t_min) / (n_frames - 1)
        n_points = txy.shape[0]
        ones_arr = torch.ones(n_points)
        u_min, v_min, p_min = float("inf"), float("inf"), float("inf")
        u_max, v_max, p_max = float("-inf"), float("-inf"), float("-inf")
        for iter in range(n_frames-1, -1, -1):
            time = t_min + t_step * iter
            if self.use_time_marching:
                model = self.model_at_time(time)

            with torch.no_grad():
                txy[:, 0] = ones_arr * time

            u_exact, v_exact, p_exact = self._ref_uvp_at_time(time)
            u_exact = u_exact.cpu() * U_wave
            v_exact = v_exact.cpu() * U_wave
            p_exact = p_exact.cpu() * P_wave

            u_model, v_model, p_model = model_txy_to_uvp(model, txy)
            u_model = u_model * U_wave
            v_model = v_model * U_wave
            p_model = p_model * P_wave

            u_min = min(u_min, u_exact.min().item(), u_model.min().item())
            v_min = min(v_min, v_exact.min().item(), v_model.min().item())
            p_min = min(p_min, p_exact.min().item(), p_model.min().item())
            u_max = max(u_max, u_exact.max().item(), u_model.max().item())
            v_max = max(v_max, v_exact.max().item(), v_model.max().item())
            p_max = max(p_max, p_exact.max().item(), p_model.max().item())
        #u_min, v_min, p_min = u_min * U_wave, v_min * U_wave, p_min * P_wave
        #u_max, v_max, p_max = u_max * U_wave, v_max * U_wave, p_max * P_wave

        # Вычисление адекватных размеров для figure
        _, h = plt.rcParams['figure.figsize']
        nrows, ncols = 4, 3
        fig_h = h * nrows * 0.6
        fig_w = fig_h * (ncols / nrows) * (x_max - x_min) / (y_max - y_min)
        fig, ((ax1, ax2, ax3), (ax4, ax5, ax6), (ax7, ax8, ax9), (ax10, ax11, ax12)) = plt.subplots(
            nrows, ncols,
            figsize=(fig_w, fig_h),
            layout="compressed"#,
            #sharex=True,
            #sharey=True
        )

        # Plot exact solution
        x_dim = txy[:, 1].detach().clone() * L_wave    # dimentional
        y_dim = txy[:, 2].detach().clone() * L_wave
        xlims_dim = [self.xlims[0]*L_wave, self.xlims[1]*L_wave]
        ylims_dim = [self.ylims[0]*L_wave, self.ylims[1]*L_wave]
        triang = tri.Triangulation(x_dim, y_dim)
        triangles = triang.triangles
        xmid = x_dim[triangles].mean(axis=1)
        ymid = y_dim[triangles].mean(axis=1)
        mask = (xmid - x_obs*L_wave)**2 + (ymid - y_obs*L_wave)**2 < (d_obs*L_wave/2.0)**2
        triang.set_mask(mask)
        placeholder_tensor = torch.zeros_like(u_exact)
        with torch.no_grad():
            pc1 = ax1.tripcolor(
                triang, placeholder_tensor, vmin=u_min, vmax=u_max, shading="gouraud", cmap="coolwarm")
            ax1.set(title="u exact")
            fig.colorbar(pc1, ax=ax1).ax.set_title("m/s", pad=8)
            pc2 = ax2.tripcolor(
                triang, placeholder_tensor, vmin=v_min, vmax=v_max, shading="gouraud", cmap="coolwarm")
            ax2.set(title="v exact")
            fig.colorbar(pc2, ax=ax2).ax.set_title("m/s", pad=8)
            pc3 = ax3.tripcolor(
                triang, placeholder_tensor, vmin=p_min, vmax=p_max, shading="gouraud", cmap="binary")
            ax3.set(title="p exact")
            fig.colorbar(pc3, ax=ax3).ax.set_title("Pa", pad=8)

            # Plot model solution
            pc4 = ax4.tripcolor(
                triang, placeholder_tensor, vmin=u_min, vmax=u_max, shading="gouraud", cmap="coolwarm")
            ax4.set(title="u model")
            fig.colorbar(pc4, ax=ax4).ax.set_title("m/s", pad=8)
            pc5 = ax5.tripcolor(
                triang, placeholder_tensor, vmin=v_min, vmax=v_max, shading="gouraud", cmap="coolwarm")
            ax5.set(title="v model")
            fig.colorbar(pc5, ax=ax5).ax.set_title("m/s", pad=8)
            pc6 = ax6.tripcolor(
                triang, placeholder_tensor, vmin=p_min, vmax=p_max, shading="gouraud", cmap="binary")
            ax6.set(title="p model")
            fig.colorbar(pc6, ax=ax6).ax.set_title("Pa", pad=8)

            # Plot error
            pc7 = ax7.tripcolor(
                triang, placeholder_tensor, shading="gouraud", cmap="binary")
            ax7.set(title="abs(u_model - u_exact)")
            fig.colorbar(pc7, ax=ax7).ax.set_title("m/s", pad=8)
            pc8 = ax8.tripcolor(
                triang, placeholder_tensor, shading="gouraud", cmap="binary")
            ax8.set(title="abs(v_model - v_exact)")
            fig.colorbar(pc8, ax=ax8).ax.set_title("m/s", pad=8)
            pc9 = ax9.tripcolor(
                triang, placeholder_tensor, shading="gouraud", cmap="binary")
            ax9.set(title="abs(p_model - p_exact)")
            fig.colorbar(pc9, ax=ax9).ax.set_title("Pa", pad=8)

            # Plot residuals
            pc10 = ax10.tripcolor(triang, placeholder_tensor, shading="gouraud", cmap="binary")
            ax10.set(title="res_u (dimentionless)")
            fig.colorbar(pc10, ax=ax10, format="%.2e")
            pc11 = ax11.tripcolor(triang, placeholder_tensor, shading="gouraud", cmap="binary")
            ax11.set(title="res_v (dimentionless)")
            fig.colorbar(pc11, ax=ax11, format="%.2e")
            pc12 = ax12.tripcolor(triang, placeholder_tensor, shading="gouraud", cmap="binary")
            ax12.set(title="res_p (dimentionless)")
            fig.colorbar(pc12, ax=ax12, format="%.2e")

            for ax in [ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8, ax9, ax10, ax11, ax12]:
                ax.set(xlabel="$x, m$", ylabel="$y, m$", xlim=xlims_dim, ylim=ylims_dim, aspect="equal")

            writer = anim.FFMpegWriter(
                fps=video_fps,
                codec='libx264',
                extra_args=[
                    "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2",
                    "-pix_fmt", "yuv420p",
                    "-preset", "ultrafast",
                    "-threads", "0"
                ]
            )
            pbar = trange(n_frames, desc="Rendering model_vs_reference video")
            with writer.saving(fig, self.results_folder / "model_vs_exact_solution.mp4", dpi=video_dpi):
                for iter in pbar:
                    time = t_min + t_step * iter
                    if self.use_time_marching:
                        model = self.model_at_time(time)

                    fig.suptitle(f"time = {time*T_wave:10.2f}")

                    # Reference solution
                    u_exact, v_exact, p_exact = self._ref_uvp_at_time(time)
                    u_exact = u_exact.cpu() * U_wave
                    v_exact = v_exact.cpu() * U_wave
                    p_exact = p_exact.cpu() * P_wave

                    # Model solution
                    txy[:, 0] = ones_arr * time
                    with torch.enable_grad():
                        u_model, v_model, p_model = model_txy_to_uvp(model, txy)
                        u_model = u_model * U_wave
                        v_model = v_model * U_wave
                        p_model = p_model * P_wave

                    # Update reference solution
                    pc1.set_array(u_exact)
                    pc2.set_array(v_exact)
                    pc3.set_array(p_exact)

                    # Update model solution
                    pc4.set_array(u_model)
                    pc5.set_array(v_model)
                    pc6.set_array(p_model)

                    # Update error
                    u_error = torch.abs(u_exact - u_model)
                    v_error = torch.abs(v_exact - v_model)
                    p_error = torch.abs(p_exact - p_model)
                    pc7.set_array(u_error)
                    pc7.set_clim(u_error.min(), u_error.max())
                    pc8.set_array(v_error)
                    pc8.set_clim(v_error.min(), v_error.max())
                    pc9.set_array(p_error)
                    pc9.set_clim(p_error.min(), p_error.max())

                    # Update residuals
                    with torch.enable_grad():
                        (
                            res_u, res_v, _, _, _, res_p
                        ) = self.compute_res_r(txy.detach().clone().requires_grad_(True), model=model)
                    pc10.set_array(res_u.squeeze().detach()); pc10.set_clim(res_u.min(), res_u.max())
                    pc11.set_array(res_v.squeeze().detach()); pc11.set_clim(res_v.min(), res_v.max())
                    pc12.set_array(res_p.squeeze().detach()); pc12.set_clim(res_p.min(), res_p.max())

                    writer.grab_frame()

        self.model.to(self.device)

Запуск обучения без каких-либо улучшений

In [ ]:
config_baseline = PINNConfig(
    # General
    results_folder = PINNConfig().results_folder / "baseline3",
    reference_solution_path = Path.cwd() / "data" / "navier-stokes_2d_incompressible_nonsteady_obstacle.pt",
    
    # Initial-condition source:
    # - "zero": use zero velocity and pressure as IC
    # - "reference": use COMSOL/reference solution at ic_reference_time
    # - "pretrained_model": use a saved model as IC target
    ic_source = "reference",
    ic_reference_time = 4.0,    # Reference-solution time, in seconds, used as the IC target when ic_source="reference"
    pretrained_ic_model_path = None,    # Path to a pretrained IC model. Used when ic_source="pretrained_model"
    #pretrained_ic_model_path = Path.cwd() / "runs/05_03_navier-stokes_2d_incompressible_nonsteady_obstacle/FFE256+CosineAnnealingLR_lr1e-4+bigger_network+grad_norm_weighting+GELU+time_marching_0-4seconds/model/model_window=4.pth",

    # Problem parameters
    xlims = (-0.4, 0.94),    # x_min, x_max
    ylims = (-0.1, 0.4),    # y_min, y_max
    tlims = (4.0, 5.0),    # t_min, t_max
    nu = 0.00103,    # кинематическая вязкость (масло касторовое), м2/с
    rho = 960,    # плотность (масло касторовое), кг/м3
    u0 = 1.0,    # начальная скорость в центре потока, м/с

    # Model parameters
    device = "cuda" if torch.cuda.is_available() else "cpu",
    init_scheme = "glorot_normal",
    layers = (3, 256, 256, 256, 256, 5),   # (x, y) -> (psi, p, sigma_xx, sigma_xy(=sigma_yx), sigma_yy)
    activation = "gelu",

    # Training parameters
    n_epochs = 10000,
    n_batches = 10,   # number of batches per epoch
    use_data_loss = False,
    lambdas = MappingProxyType({
        "u_r": 1.0, "v_r": 1.0, "sigma_xx_r": 1.0, "sigma_xy_r": 1.0, "sigma_yy_r": 1.0, "p_r": 1.0,
        "u_ic": 1.0, "v_ic": 1.0, "p_ic": 1.0,
        "u_walls": 1.0, "v_walls": 1.0, "u_in": 1.0, "v_in": 1.0,
        "sigma_xx_out": 1.0, "sigma_xy_out": 1.0,
        #"u_data": 1.0, "v_data": 1.0, "p_data": 1.0
        "u_data": 1.0, "v_data": 1.0
    }),
    lr = 1e-3,    # initial learning rate
    logging_freq = 200,    # log metrics once every N gradient descent steps

    # Sampling
    use_static_training_set = False,
    use_adaptive_sampling = False,
    resampling_frequency = 20,
    n_r = 100000,
    data_fraction = 0.1,

    # Log metrics (like loss and error) during training for different t intervals
    use_temporal_logging = False,
    temporal_logging_freq = 500,    # log metrics once every N gradient descent steps
    temporal_logging_nbins = 10,    # Slice time into this number of equal segments

    # Checkpointing
    use_checkpointing = True,
    checkpointing_freq = 60000,    # save model and metrics to the results_folder once every N gradient descent steps

    # Fourier Feature embedding
    use_FFE = True,
    FFE_m = 256,
    FFE_sigma = 1.0,
    FFE_keep_dims = None,    # Indices of input dimensions to be passed through unchanged

    # Grad norm weighting
    use_grad_norm_weighting = False,
    grad_norm_weighting_alpha = 0.9,
    grad_norm_weighting_freq = 1000,    # update lambdas once every N gradient descent steps

    # Causal weighting
    use_causal_weighting_scheme = False,
    causal_m = 16,
    causal_eps = (0.01, 0.1, 1.0, 10.0, 100.0, 500.0, 1000.0),
    W_logging_freq = 500,    # log temporal weights and L_t once every N gradient descent steps

    # Time marching
    use_time_marching = False,
    n_time_windows = 8,

    # Video parameters
    render_video = False,
    video_fps = 10,
    video_dpi = 100,
    video_render_freq = 500    # render a frame once every N gradient descent steps
)

model_baseline = MultilayerPerceptronWithFFE(
   config_baseline.layers, config_baseline.init_scheme, activation=config_baseline.activation,
   use_FFE=config_baseline.use_FFE, FFE_m=config_baseline.FFE_m,
   FFE_sigma=config_baseline.FFE_sigma, FFE_keep_dims=config_baseline.FFE_keep_dims)

def optimizer_factory(model):
    return torch.optim.Adam(model.parameters(), lr=config_baseline.lr)

def scheduler_factory(optimizer):
    T_max = config_baseline.n_epochs*config_baseline.n_batches
    if config_baseline.use_time_marching:
        T_max = T_max // config_baseline.n_time_windows
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_max, eta_min=1e-4)

trainer_baseline = Trainer(model_baseline, optimizer_factory, scheduler_factory, cfg=config_baseline)
print(config_baseline)
metrics_baseline = trainer_baseline.train()

trainer_baseline.render_metrics_plots(metrics_baseline, save_to_disk=True)
trainer_baseline.render_model_vs_reference_video()

C:\Users\Alexey\AppData\Local\Temp\ipykernel_65596\174639086.py:95: RuntimeWarning: When using both use_data_loss and use_time_marching, some time windows may contain no data points, or too few data points to make a data batch. Both cases currently result in an exception.
  trainer_baseline = Trainer(model_baseline, optimizer_factory, scheduler_factory, cfg=config_baseline)


GENERAL
results_folder: d:\Учёба\adf\Скрипты\NN_pytorch_BVP\runs\05_03_navier-stokes_2d_incompressible_nonsteady_obstacle\baseline3
reference_solution_path: d:\Учёба\adf\Скрипты\NN_pytorch_BVP\data\navier-stokes_2d_incompressible_nonsteady_obstacle.pt
pretrained_ic_model_path: None
use_temporal_logging: False
temporal_logging_freq: 500
temporal_logging_nbins: 10

PROBLEM PARAMETERS
xlims: (-0.4, 0.94)
ylims: (-0.1, 0.4)
tlims: (0.0, 5.0)
coords_obs: (-0.08750000000000002, 0.15)
d_obs: 0.125
nu: 0.00103
rho: 960
u0: 1.0

DIMENSIONLESS PARAMETERS
L_wave: 0.125
U_wave: 0.6666666666666666
T_wave: 0.1875
P_wave: 426.66666666666663
F_wave: 3.5555555555555554
Re: 80.9061488673139
Sigma_wave: 426.66666666666663

MODEL PARAMETERS
device: cuda
init_scheme: glorot_normal
layers: (3, 256, 256, 256, 256, 5)
activation: gelu

TRAINING PARAMETERS
n_epochs: 20000
n_batches: 10
use_data_loss: True
lambdas: {'u_r': 1.0, 'v_r': 1.0, 'sigma_xx_r': 1.0, 'sigma_xy_r': 1.0, 'sigma_yy_r': 1.0, 'p_r': 1.0, 'u_

Window 1/8:   0%|          | 0/200000 [00:00<?, ?it/s]    C:\Users\Alexey\AppData\Local\Temp\ipykernel_65596\3310056174.py:733: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at C:/actions-runner/_work/pytorch/pytorch/pytorch/aten/src\ATen/native/BucketizationUtils.h:34.)
  bin_idx = torch.bucketize(txy_r[:, 0].detach(), edges[1:-1])      # к какому временнОму интервалу принадлежит каждая точка
Window 1/8:   0%|          | 28/200000 [00:10<18:53:07,  2.94it/s, loss=4.4, rel_err_u_l2=100, rel_err_v_l2=160, rel_err_p_l2=100, causal_w_min=0.953, causal_eps=0.1]

KeyboardInterrupt: 

In [ ]:
if is_in_Colab:
    from google.colab import runtime
    runtime.unassign()